In [ ]:
# ==========================================
# 0. INSTALLATION
# ==========================================
!pip install -q -U pymupdf4llm unsloth trl datasets accelerate

import torch
import json
import os
import gc
import pymupdf4llm 
from unsloth import FastLanguageModel
from langchain.text_splitter import RecursiveCharacterTextSplitter
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq

# --- CONFIGURATION FOR 2x T4 ---
MODEL_PATH = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" 
INPUT_PDF = "/kaggle/input/the-gale-encyclopedia-of-medicine-3rd-edition/The-Gale-Encyclopedia-of-Medicine-3rd-Edition-staibabussalamsula.ac_.id_.pdf" 
DATA_FILE = "medical_finetune_data.json"
OUTPUT_DIR = "/kaggle/working/qwen_medical_finetuned"

# Check available GPUs
print(f"Available GPUs: {torch.cuda.device_count()}")
print(f"GPU 0: {torch.cuda.get_device_name(0)}")
if torch.cuda.device_count() > 1:
    print(f"GPU 1: {torch.cuda.get_device_name(1)}")

# ==========================================
# PART 1: GENERATE DATA (USING LOCAL QWEN)
# ==========================================
print("\n=== PART 1: GENERATING DATA WITH QWEN ===")

# 1. Load Model for Inference (uses first GPU)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# 2. Convert PDF to Markdown Text
print(f"Reading PDF: {INPUT_PDF}...")
if os.path.exists(INPUT_PDF):
    text_content = pymupdf4llm.to_markdown(INPUT_PDF)
else:
    print("Warning: PDF not found. Using dummy text for testing.")
    text_content = "Acute bronchitis is inflammation of the bronchi. Symptoms include cough and phlegm."

# 3. Split Text into Chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = splitter.split_text(text_content)
print(f"Split PDF into {len(chunks)} chunks.")

# 4. Generate Q&A with Multiple Question Types
dataset = []
print("Generating Q&A pairs... (Using GPU 0)")

def create_diverse_prompt(text_chunk, idx):
    question_types = [
        "definition: Create a question asking 'What is...' about a medical term",
        "symptom: Create a question asking about symptoms of a condition",
        "treatment: Create a question asking about treatment options",
        "cause: Create a question asking about causes or risk factors",
        "diagnosis: Create a question asking how a condition is diagnosed",
        "prevention: Create a question asking about prevention methods"
    ]
    
    q_type = question_types[idx % len(question_types)]
    
    return f"""<|im_start|>system
You are a medical education expert creating diverse question-answer pairs.<|im_end|>
<|im_start|>user
Read this medical text carefully. Create ONE high-quality question-answer pair.

Question Type: {q_type}

Rules:
1. Question must be clear, specific, and answerable from the text
2. Answer must be comprehensive but concise (2-4 sentences)
3. Use proper medical terminology
4. Output valid JSON with keys: "instruction", "input", "output"
5. Set "input" to empty string ""

TEXT:
{text_chunk}

Output JSON only:<|im_end|>
<|im_start|>assistant
{{"instruction": """

# Process more chunks with T4's capacity
max_chunks = min(len(chunks), 500)

for i, chunk in enumerate(chunks[:max_chunks]):
    if len(chunk.strip()) < 100:
        continue
        
    prompt = create_diverse_prompt(chunk, i)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=300,
        use_cache=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1
    )
    
    decoded = tokenizer.batch_decode(outputs)[0]
    response = decoded.split("<|im_start|>assistant")[-1].replace("<|im_end|>", "").strip()
    
    try:
        start_idx = response.find("{")
        end_idx = response.rfind("}") + 1
        if start_idx != -1 and end_idx > start_idx:
            json_str = response[start_idx:end_idx]
            qa = json.loads(json_str)
            
            if (qa.get("instruction") and qa.get("output") and 
                len(qa["instruction"]) > 10 and len(qa["output"]) > 20):
                qa["input"] = ""
                dataset.append(qa)
                
                if (i + 1) % 50 == 0:
                    print(f"Generated {len(dataset)} valid pairs from {i+1} chunks")
    except Exception as e:
        if i < 10:  # Only show first few errors
            print(f"Chunk {i+1}: Failed ({str(e)[:50]})")

with open(DATA_FILE, 'w') as f:
    json.dump(dataset, f, indent=4)
print(f"\n✓ Saved {len(dataset)} high-quality examples to {DATA_FILE}")

# ==========================================
# CRITICAL: CLEANUP MEMORY
# ==========================================
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print("\n✓ Memory cleared for training phase.")

# ==========================================
# PART 2: FINE-TUNE (OPTIMIZED FOR 2x T4)
# ==========================================
print("\n=== PART 2: FINE-TUNING QWEN ON 2x T4 GPUs ===")

# 1. Reload Model Fresh for Training
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    device_map = "auto",  # Automatically distribute across GPUs
)

# 2. Enhanced LoRA Configuration for T4
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,  # Good capacity for medical adaptation
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,
)

print("✓ LoRA adapters configured")
print(f"✓ Model distributed across {torch.cuda.device_count()} GPUs")

# 3. Load and validate dataset
data = load_dataset("json", data_files=DATA_FILE, split="train")
print(f"✓ Loaded {len(data)} training examples")

def format_chatml(examples):
    texts = []
    for instr, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        text = f"""<|im_start|>system
You are a knowledgeable medical assistant providing accurate, evidence-based information.<|im_end|>
<|im_start|>user
{instr}<|im_end|>
<|im_start|>assistant
{out}<|im_end|>"""
        texts.append(text)
    return {"text": texts}

data = data.map(format_chatml, batched=True, remove_columns=data.column_names)
print("✓ Dataset formatted for ChatML")

# 4. OPTIMIZED TRAINER FOR 2x T4 GPUs
# Key optimization: Higher effective batch size with gradient accumulation
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = data,
    dataset_text_field = "text",
    max_seq_length = 2048,
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc = 4,  # Increased for faster data loading with 2 GPUs
    packing = False,
    args = TrainingArguments(
        # OPTIMIZED BATCH SIZE FOR 2x T4 (16GB each)
        per_device_train_batch_size = 4,  # 4 per GPU = 8 total
        gradient_accumulation_steps = 4,   # Effective batch = 8 × 4 = 32
        
        # Training duration
        num_train_epochs = 3,
        warmup_ratio = 0.05,
        
        # Learning configuration
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        weight_decay = 0.01,
        optim = "adamw_8bit",
        
        # Precision settings for T4
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        
        # Multi-GPU specific settings
        ddp_find_unused_parameters = False,  # Faster with LoRA
        dataloader_num_workers = 4,  # Parallel data loading
        dataloader_pin_memory = True,  # Faster host-to-GPU transfer
        
        # Logging and saving
        logging_steps = 5,
        save_strategy = "epoch",
        save_total_limit = 2,
        output_dir = "outputs",
        
        # Performance optimizations
        gradient_checkpointing = True,
        max_grad_norm = 1.0,  # Gradient clipping for stability
        
        seed = 3407,
        report_to = "none",
    ),
)

print("✓ Trainer configured for 2x T4 GPUs")
print(f"   • Per-device batch size: 4")
print(f"   • Total batch size per step: 8 (4 per GPU × 2 GPUs)")
print(f"   • Gradient accumulation: 4 steps")
print(f"   • Effective batch size: 32")
print(f"   • Expected training time: ~{len(data) * 3 // 32 // 60} minutes for 3 epochs")

# 5. Train with progress monitoring
print("\n🚀 Starting training on 2x T4 GPUs...")
trainer.train()
print("✓ Training complete!")

# 6. Save Final Model
print(f"\n💾 Saving model to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save merged version for deployment
print("💾 Saving merged model (16-bit)...")
model.save_pretrained_merged(
    f"{OUTPUT_DIR}_merged_16bit",
    tokenizer,
    save_method = "merged_16bit"
)

# Optional: Save 4-bit quantized version for efficient inference
print("💾 Saving 4-bit quantized model for inference...")
model.save_pretrained_merged(
    f"{OUTPUT_DIR}_merged_4bit",
    tokenizer,
    save_method = "merged_4bit"
)

print("\n✅ ALL DONE! Model saved in three formats:")
print(f"   1. LoRA adapters: {OUTPUT_DIR}")
print(f"   2. Merged 16-bit: {OUTPUT_DIR}_merged_16bit")
print(f"   3. Merged 4-bit: {OUTPUT_DIR}_merged_4bit (best for inference)")


In [ ]:
!pip install -q -U pymupdf4llm unsloth trl datasets accelerate psutil

In [ ]:
# ==========================================
# 0. INSTALLATION
# ==========================================
# !pip install -q -U pymupdf4llm unsloth trl datasets accelerate psutil

import torch
import json
import os
import gc
import pymupdf4llm 
from unsloth import FastLanguageModel
from langchain.text_splitter import RecursiveCharacterTextSplitter
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
import time
from datetime import datetime

# --- CONFIGURATION FOR 4500-PAGE PDF ---
MODEL_PATH = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"  # 8B for faster processing

INPUT_PDF = "/kaggle/input/the-gale-encyclopedia-of-medicine-3rd-edition/The-Gale-Encyclopedia-of-Medicine-3rd-Edition-staibabussalamsula.ac_.id_.pdf" 
DATA_FILE = "medical_finetune_data.json"
OUTPUT_DIR = "/kaggle/working/llama3_medical_encyclopedia"

# IMPORTANT: Control how much to process
MAX_CHUNKS_TO_PROCESS = 3000  # Adjust based on time: 1000=30min, 3000=90min, 6000=3hrs
SAVE_CHECKPOINT_EVERY = 500   # Save progress every N chunks

print(f"🔍 Configuration:")
print(f"   • Max chunks to process: {MAX_CHUNKS_TO_PROCESS}")
print(f"   • Checkpoint interval: {SAVE_CHECKPOINT_EVERY}")
print(f"   • Estimated time: {MAX_CHUNKS_TO_PROCESS * 0.06:.0f}-{MAX_CHUNKS_TO_PROCESS * 0.08:.0f} minutes")
print(f"   • Expected Q&A pairs: ~{int(MAX_CHUNKS_TO_PROCESS * 0.75)}")

# ==========================================
# PART 1: GENERATE DATA FROM 4500-PAGE PDF
# ==========================================
print("\n" + "="*70)
print("PART 1: PROCESSING 4500-PAGE MEDICAL ENCYCLOPEDIA")
print("="*70)

start_time = time.time()

# 1. Load Model for Inference
print("\n📥 Loading Llama 3.1 8B for data generation...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
print("✓ Model loaded")

# 2. Convert Large PDF to Text (with progress indicator)
print(f"\n📄 Reading 4500-page PDF: {INPUT_PDF}")
print("   ⏳ This may take 2-5 minutes for large PDF...")

if os.path.exists(INPUT_PDF):
    pdf_start = time.time()
    text_content = pymupdf4llm.to_markdown(INPUT_PDF)
    pdf_time = time.time() - pdf_start
    
    print(f"✓ PDF converted in {pdf_time/60:.1f} minutes")
    print(f"   • Total characters: {len(text_content):,}")
    print(f"   • Estimated pages: ~{len(text_content) // 2500}")
else:
    raise FileNotFoundError(f"PDF not found at {INPUT_PDF}")

# 3. Intelligent Chunking for Medical Content
print("\n✂️  Splitting into optimized chunks...")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Good for medical paragraphs
    chunk_overlap=200,  # Higher overlap for medical context
    separators=[
        "\n\n# ",  # Major sections
        "\n\n## ", # Sub-sections  
        "\n\n",    # Paragraphs
        "\n",      # Lines
        ". ",      # Sentences
        " "        # Words
    ],
    length_function=len,
)

chunks = splitter.split_text(text_content)
total_chunks = len(chunks)

print(f"✓ Created {total_chunks:,} chunks from 4500 pages")
print(f"   • Processing {min(MAX_CHUNKS_TO_PROCESS, total_chunks):,} chunks")
print(f"   • Skipping {max(0, total_chunks - MAX_CHUNKS_TO_PROCESS):,} chunks")

# 4. Generate Q&A with Checkpointing for Long Runs
dataset = []
failed_chunks = 0
chunks_to_process = min(MAX_CHUNKS_TO_PROCESS, total_chunks)

print("\n🤖 Generating medical Q&A pairs...")
print("   💾 Progress will be auto-saved every 500 chunks")

# Load existing progress if available
if os.path.exists(DATA_FILE):
    try:
        with open(DATA_FILE, 'r') as f:
            dataset = json.load(f)
        print(f"   📂 Resumed: Found {len(dataset)} existing Q&A pairs")
        start_idx = len(dataset)
    except:
        start_idx = 0
else:
    start_idx = 0

# Medical question types
def create_medical_prompt(text_chunk, idx):
    question_types = [
        ("definition", "Create a question asking 'What is...' about a medical term or condition"),
        ("symptoms", "Create a question about symptoms or clinical presentation"),
        ("treatment", "Create a question about treatment options or medications"),
        ("causes", "Create a question about causes, etiology, or risk factors"),
        ("diagnosis", "Create a question about diagnostic methods or tests"),
        ("prevention", "Create a question about prevention or lifestyle modifications"),
        ("complications", "Create a question about complications or prognosis"),
        ("epidemiology", "Create a question about prevalence, demographics, or statistics"),
    ]
    
    q_type, q_instruction = question_types[idx % len(question_types)]
    
    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a medical education expert creating precise question-answer pairs from medical literature.<|eot_id|><|start_header_id|>user<|end_header_id|>

Read this medical text and create ONE high-quality question-answer pair.

Type: {q_type}
Task: {q_instruction}

Requirements:
- Question must be specific and directly answerable from the text
- Answer must be comprehensive (2-5 sentences) with proper medical terminology
- Output ONLY valid JSON: {{"instruction": "question here", "input": "", "output": "answer here"}}
- No extra text before or after JSON

TEXT:
{text_chunk}

JSON:<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{{"""

# Progress tracking variables
last_checkpoint_time = time.time()
generation_start = time.time()

# Process chunks with progress tracking
for i in range(start_idx, chunks_to_process):
    chunk = chunks[i]
    
    # Skip very short chunks
    if len(chunk.strip()) < 150:
        continue
    
    try:
        prompt = create_medical_prompt(chunk, i)
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        
        # Generate with medical-optimized parameters
        outputs = model.generate(
            **inputs, 
            max_new_tokens=400,
            use_cache=True,
            temperature=0.6,  # Lower for factual medical content
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.15,
            do_sample=True,
        )
        
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]
        
        # Extract assistant response
        if "<|start_header_id|>assistant<|end_header_id|>" in decoded:
            response = decoded.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
            response = response.replace("<|eot_id|>", "").replace("<|end_of_text|>", "").strip()
        else:
            response = decoded
        
        # Extract JSON
        start_idx_json = response.find("{")
        end_idx_json = response.rfind("}") + 1
        
        if start_idx_json != -1 and end_idx_json > start_idx_json:
            json_str = response[start_idx_json:end_idx_json]
            qa = json.loads(json_str)
            
            # Quality validation
            if (qa.get("instruction") and qa.get("output") and 
                len(qa["instruction"]) > 15 and 
                len(qa["output"]) > 30 and
                "?" in qa["instruction"]):
                
                qa["input"] = ""
                dataset.append(qa)
                
        else:
            failed_chunks += 1
            
    except Exception as e:
        failed_chunks += 1
        if failed_chunks <= 3:
            print(f"   ⚠️  Chunk {i+1} failed: {str(e)[:50]}")
    
    # Progress reporting every 100 chunks
    if (i + 1) % 100 == 0:
        elapsed = time.time() - generation_start
        rate = (i + 1 - start_idx) / elapsed
        remaining = (chunks_to_process - i - 1) / rate if rate > 0 else 0
        
        print(f"   📊 Progress: {i+1}/{chunks_to_process} chunks | "
              f"✓ {len(dataset)} Q&A pairs | "
              f"✗ {failed_chunks} failed | "
              f"⏱️  ETA: {remaining/60:.0f} min")
    
    # Auto-save checkpoint every N chunks
    if (i + 1) % SAVE_CHECKPOINT_EVERY == 0:
        checkpoint_file = f"{DATA_FILE}.checkpoint"
        with open(checkpoint_file, 'w', encoding='utf-8') as f:
            json.dump(dataset, f, indent=2, ensure_ascii=False)
        checkpoint_time = time.time() - last_checkpoint_time
        print(f"   💾 Checkpoint saved: {len(dataset)} pairs "
              f"(last {SAVE_CHECKPOINT_EVERY} in {checkpoint_time/60:.1f} min)")
        last_checkpoint_time = time.time()
    
    # Periodic memory cleanup
    if (i + 1) % 200 == 0:
        torch.cuda.empty_cache()

# Final save
print(f"\n💾 Saving final dataset...")
with open(DATA_FILE, 'w', encoding='utf-8') as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)

generation_time = time.time() - generation_start
success_rate = (len(dataset) / chunks_to_process) * 100 if chunks_to_process > 0 else 0

print(f"\n✅ Data Generation Complete!")
print(f"   • Total Q&A pairs: {len(dataset):,}")
print(f"   • Success rate: {success_rate:.1f}%")
print(f"   • Generation time: {generation_time/60:.1f} minutes")
print(f"   • Rate: {len(dataset) / (generation_time/60):.0f} pairs/minute")
print(f"   • Saved to: {DATA_FILE}")

# Quality check
if len(dataset) < 100:
    print(f"\n⚠️  WARNING: Only {len(dataset)} Q&A pairs generated.")
    print("   Consider increasing MAX_CHUNKS_TO_PROCESS for better model quality.")
elif len(dataset) < 500:
    print(f"\n✓ Dataset size is adequate for basic fine-tuning")
elif len(dataset) < 2000:
    print(f"\n✓ Dataset size is good for quality fine-tuning")
else:
    print(f"\n🌟 Excellent dataset size for comprehensive medical training!")

# ==========================================
# CLEANUP MEMORY
# ==========================================
print("\n🧹 Clearing memory...")
del model
del tokenizer
del inputs
del outputs
del chunks
del text_content
gc.collect()
torch.cuda.empty_cache()
print("✓ Memory cleared\n")

# ==========================================
# PART 2: FINE-TUNE WITH GENERATED DATA
# ==========================================
print("="*70)
print("PART 2: FINE-TUNING LLAMA 3.1 8B ON MEDICAL ENCYCLOPEDIA")
print("="*70)

# 1. Reload Model for Training
print("\n📥 Loading model for training...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    device_map = "auto",
)
print("✓ Model loaded")

# 2. Configure LoRA
print("\n⚙️  Configuring LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,  # Good capacity for 8B model
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,
)
print("✓ LoRA configured")

# 3. Load Dataset
print(f"\n📊 Loading dataset...")
data = load_dataset("json", data_files=DATA_FILE, split="train")
print(f"✓ Loaded {len(data):,} training examples")

# Format for training
def format_llama_chat(examples):
    texts = []
    for instr, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a medical assistant trained on the Gale Encyclopedia of Medicine, providing accurate evidence-based information.<|eot_id|><|start_header_id|>user<|end_header_id|>

{instr}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{out}<|eot_id|>"""
        texts.append(text)
    return {"text": texts}

print("🔄 Formatting dataset...")
data = data.map(format_llama_chat, batched=True, remove_columns=data.column_names)
print("✓ Dataset formatted")

# 4. Configure Trainer (Optimized for 2x T4)
print("\n⚙️  Configuring trainer...")

# Dynamic epoch calculation based on dataset size
if len(data) < 500:
    num_epochs = 5  # More epochs for small datasets
elif len(data) < 2000:
    num_epochs = 3  # Standard
else:
    num_epochs = 2  # Fewer epochs for large datasets to avoid overfitting

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = data,
    dataset_text_field = "text",
    max_seq_length = 2048,
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc = 4,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,  # 8B model can handle 4
        gradient_accumulation_steps = 4,  # Effective batch = 32
        num_train_epochs = num_epochs,
        warmup_ratio = 0.03,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        max_grad_norm = 0.3,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        save_strategy = "epoch",
        save_total_limit = 2,
        output_dir = "outputs",
        report_to = "none",
        seed = 3407,
    ),
)

print("✓ Trainer configured")
print(f"\n📊 Training Configuration:")
print(f"   • Dataset size: {len(data):,} examples")
print(f"   • Epochs: {num_epochs}")
print(f"   • Batch size: 4 per GPU × 2 GPUs = 8")
print(f"   • Gradient accumulation: 4 (effective batch = 32)")
print(f"   • Total training steps: {len(data) * num_epochs // 32}")
print(f"   • Estimated time: ~{len(data) * num_epochs / 32 * 0.5:.0f} minutes")

# 5. Train
print("\n" + "="*70)
print("🚀 STARTING TRAINING")
print("="*70 + "\n")

train_start = time.time()
trainer_stats = trainer.train()
train_time = time.time() - train_start

print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)
print(f"\n📈 Training Results:")
print(f"   • Final loss: {trainer_stats.metrics['train_loss']:.4f}")
print(f"   • Training time: {train_time/60:.1f} minutes")
print(f"   • Steps completed: {trainer_stats.global_step}")

# 6. Save Models
print(f"\n💾 Saving models...")

# LoRA adapters
print("   1/3 Saving LoRA adapters...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Merged 16-bit
print("   2/3 Saving merged 16-bit...")
model.save_pretrained_merged(
    f"{OUTPUT_DIR}_merged_16bit",
    tokenizer,
    save_method = "merged_16bit"
)

# Merged 4-bit (best for inference)
print("   3/3 Saving merged 4-bit...")
model.save_pretrained_merged(
    f"{OUTPUT_DIR}_merged_4bit",
    tokenizer,
    save_method = "merged_4bit"
)

# Save metadata
training_info = {
    "source": "Gale Encyclopedia of Medicine (4500 pages)",
    "base_model": MODEL_PATH,
    "training_examples": len(data),
    "chunks_processed": chunks_to_process,
    "epochs": num_epochs,
    "final_loss": float(trainer_stats.metrics['train_loss']),
    "training_time_minutes": train_time / 60,
    "generation_time_minutes": generation_time / 60,
    "total_time_minutes": (time.time() - start_time) / 60,
    "date_trained": datetime.now().isoformat(),
}

with open(f"{OUTPUT_DIR}/training_info.json", 'w') as f:
    json.dump(training_info, f, indent=2)

# Final summary
total_time = time.time() - start_time
print("\n" + "="*70)
print("🎉 MEDICAL ENCYCLOPEDIA MODEL COMPLETE!")
print("="*70)
print(f"\n📦 Model Outputs:")
print(f"   1. LoRA adapters: {OUTPUT_DIR}/")
print(f"   2. Merged 16-bit: {OUTPUT_DIR}_merged_16bit/")
print(f"   3. Merged 4-bit: {OUTPUT_DIR}_merged_4bit/ ⭐")
print(f"\n📊 Final Statistics:")
print(f"   • Source: 4500-page medical encyclopedia")
print(f"   • Training examples: {len(data):,} Q&A pairs")
print(f"   • Total processing time: {total_time/60:.1f} minutes ({total_time/3600:.1f} hours)")
print(f"   • Model quality: {'Excellent' if len(data) > 2000 else 'Very Good' if len(data) > 500 else 'Good'}")
print(f"\n💡 Next Steps:")
print(f"   • Upload to HuggingFace: huggingface-cli upload YOUR_USERNAME/medical-8b {OUTPUT_DIR}_merged_4bit")
print(f"   • Test locally: See inference code examples")
print(f"   • Deploy API: Use HF Inference API or Spaces")
print(f"\n🏥 Ready to answer medical questions!")


In [ ]:
!pip install -q psutil

In [ ]:

# ==========================================
# PART 2: FINE-TUNE WITH GENERATED DATA
# ==========================================
print("="*70)
print("PART 2: FINE-TUNING LLAMA 3.1 8B ON MEDICAL ENCYCLOPEDIA")
print("="*70)

# 1. Reload Model for Training
print("\n📥 Loading model for training...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    device_map = "auto",
)
print("✓ Model loaded")

# 2. Configure LoRA
print("\n⚙️  Configuring LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,  # Good capacity for 8B model
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,
)
print("✓ LoRA configured")

# 3. Load Dataset
print(f"\n📊 Loading dataset...")
data = load_dataset("json", data_files="/kaggle/input/medical-finetune-data/medical_finetune_data_final.json", split="train")
print(f"✓ Loaded {len(data):,} training examples")

# Format for training
def format_llama_chat(examples):
    texts = []
    for instr, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a medical assistant trained on the Gale Encyclopedia of Medicine, providing accurate evidence-based information.<|eot_id|><|start_header_id|>user<|end_header_id|>

{instr}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{out}<|eot_id|>"""
        texts.append(text)
    return {"text": texts}

print("🔄 Formatting dataset...")
data = data.map(format_llama_chat, batched=True, remove_columns=data.column_names)
print("✓ Dataset formatted")

# 4. Configure Trainer (Optimized for 2x T4)
print("\n⚙️  Configuring trainer...")

# Dynamic epoch calculation based on dataset size
if len(data) < 500:
    num_epochs = 5  # More epochs for small datasets
elif len(data) < 2000:
    num_epochs = 3  # Standard
else:
    num_epochs = 2  # Fewer epochs for large datasets to avoid overfitting

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = data,
    dataset_text_field = "text",
    max_seq_length = 2048,
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc = 4,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,  # 8B model can handle 4
        gradient_accumulation_steps = 4,  # Effective batch = 32
        num_train_epochs = num_epochs,
        warmup_ratio = 0.03,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        max_grad_norm = 0.3,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        save_strategy = "epoch",
        save_total_limit = 2,
        output_dir = "outputs",
        report_to = "none",
        seed = 3407,
    ),
)

print("✓ Trainer configured")
print(f"\n📊 Training Configuration:")
print(f"   • Dataset size: {len(data):,} examples")
print(f"   • Epochs: {num_epochs}")
print(f"   • Batch size: 4 per GPU × 2 GPUs = 8")
print(f"   • Gradient accumulation: 4 (effective batch = 32)")
print(f"   • Total training steps: {len(data) * num_epochs // 32}")
print(f"   • Estimated time: ~{len(data) * num_epochs / 32 * 0.5:.0f} minutes")

# 5. Train
print("\n" + "="*70)
print("🚀 STARTING TRAINING")
print("="*70 + "\n")

train_start = time.time()
trainer_stats = trainer.train()
train_time = time.time() - train_start

print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)
print(f"\n📈 Training Results:")
print(f"   • Final loss: {trainer_stats.metrics['train_loss']:.4f}")
print(f"   • Training time: {train_time/60:.1f} minutes")
print(f"   • Steps completed: {trainer_stats.global_step}")

# 6. Save Models
print(f"\n💾 Saving models...")

# LoRA adapters
print("   1/3 Saving LoRA adapters...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Merged 16-bit
print("   2/3 Saving merged 16-bit...")
model.save_pretrained_merged(
    f"{OUTPUT_DIR}_merged_16bit",
    tokenizer,
    save_method = "merged_16bit"
)

# Merged 4-bit (best for inference)
print("   3/3 Saving merged 4-bit...")
model.save_pretrained_merged(
    f"{OUTPUT_DIR}_merged_4bit",
    tokenizer,
    save_method = "merged_4bit"
)

# Save metadata
training_info = {
    "source": "Gale Encyclopedia of Medicine (4500 pages)",
    "base_model": MODEL_PATH,
    "training_examples": len(data),
    "chunks_processed": chunks_to_process,
    "epochs": num_epochs,
    "final_loss": float(trainer_stats.metrics['train_loss']),
    "training_time_minutes": train_time / 60,
    "generation_time_minutes": generation_time / 60,
    "total_time_minutes": (time.time() - start_time) / 60,
    "date_trained": datetime.now().isoformat(),
}

with open(f"{OUTPUT_DIR}/training_info.json", 'w') as f:
    json.dump(training_info, f, indent=2)

# Final summary
total_time = time.time() - start_time
print("\n" + "="*70)
print("🎉 MEDICAL ENCYCLOPEDIA MODEL COMPLETE!")
print("="*70)
print(f"\n📦 Model Outputs:")
print(f"   1. LoRA adapters: {OUTPUT_DIR}/")
print(f"   2. Merged 16-bit: {OUTPUT_DIR}_merged_16bit/")
print(f"   3. Merged 4-bit: {OUTPUT_DIR}_merged_4bit/ ⭐")
print(f"\n📊 Final Statistics:")
print(f"   • Source: 4500-page medical encyclopedia")
print(f"   • Training examples: {len(data):,} Q&A pairs")
print(f"   • Total processing time: {total_time/60:.1f} minutes ({total_time/3600:.1f} hours)")
print(f"   • Model quality: {'Excellent' if len(data) > 2000 else 'Very Good' if len(data) > 500 else 'Good'}")
print(f"\n💡 Next Steps:")
print(f"   • Upload to HuggingFace: huggingface-cli upload YOUR_USERNAME/medical-8b {OUTPUT_DIR}_merged_4bit")
print(f"   • Test locally: See inference code examples")
print(f"   • Deploy API: Use HF Inference API or Spaces")
print(f"\n🏥 Ready to answer medical questions!")


In [1]:
!pip install -q -U pymupdf4llm unsloth trl datasets accelerate psutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.9/66.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.1/376.1 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 32.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 72.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.7/290.7 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [2]:
# ==========================================
# 0. INSTALLATION & IMPORTS
# ==========================================
import sys
import subprocess

# # Install missing package
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "psutil"])

# Now import everything
import torch
import json
import os
import gc
import psutil  # Import after installation
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
import time
from datetime import datetime

print("✅ All packages loaded successfully!")

# --- CONFIGURATION ---
MODEL_PATH = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
DATA_FILE = "/kaggle/input/medical-finetune-data/medical_finetune_data_final.json"
OUTPUT_DIR = "/kaggle/working/llama3_medical_encyclopedia"

print(f"\n✅ You have {len(json.load(open(DATA_FILE)))} Q&A pairs ready to train!")

# ==========================================
# FINE-TUNING
# ==========================================
print("\n" + "="*70)
print("FINE-TUNING LLAMA 3.1 8B ON YOUR MEDICAL DATA")
print("="*70)

start_time = time.time()

# 1. Load Model
print("\n📥 Loading model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    device_map = "auto",
)
print("✓ Model loaded")

# 2. Configure LoRA
print("\n⚙️  Configuring LoRA...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,
)
print("✓ LoRA configured")

# 3. Load Dataset
print(f"\n📊 Loading dataset...")
data = load_dataset("json", data_files=DATA_FILE, split="train")
print(f"✓ Loaded {len(data):,} examples")

# 4. Format Data
def format_llama_chat(examples):
    texts = []
    for instr, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a medical assistant trained on the Gale Encyclopedia of Medicine.<|eot_id|><|start_header_id|>user<|end_header_id|>

{instr}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{out}<|eot_id|>"""
        texts.append(text)
    return {"text": texts}

print("🔄 Formatting...")
data = data.map(format_llama_chat, batched=True, remove_columns=data.column_names)
print("✓ Formatted")

# 5. Configure Trainer - SIMPLIFIED VERSION
print("\n⚙️  Configuring trainer...")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = data,
    dataset_text_field = "text",
    max_seq_length = 2048,
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "epoch",
        save_total_limit = 2,
        report_to = "none",
    ),
)

print("✓ Trainer ready")
print(f"\n📊 Training: {len(data)} examples × 3 epochs")
print(f"   Estimated time: ~{len(data) * 3 / 32 * 0.5:.0f} minutes\n")

# 6. Train
print("="*70)
print("🚀 TRAINING STARTED")
print("="*70 + "\n")

train_start = time.time()
trainer_stats = trainer.train()
train_time = time.time() - train_start

print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)
print(f"\n📈 Results:")
print(f"   • Loss: {trainer_stats.metrics['train_loss']:.4f}")
print(f"   • Time: {train_time/60:.1f} minutes")

# 7. Save Models
print(f"\n💾 Saving models...")

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"   ✓ LoRA: {OUTPUT_DIR}")

model.save_pretrained_merged(f"{OUTPUT_DIR}_merged_16bit", tokenizer, save_method="merged_16bit")
print(f"   ✓ 16-bit: {OUTPUT_DIR}_merged_16bit")

model.save_pretrained_merged(f"{OUTPUT_DIR}_merged_4bit", tokenizer, save_method="merged_4bit")
print(f"   ✓ 4-bit: {OUTPUT_DIR}_merged_4bit ⭐")

# Metadata
training_info = {
    "model": "Llama-3.1-8B-Instruct",
    "data_source": "Gale Encyclopedia of Medicine (4500 pages)",
    "training_examples": len(data),
    "epochs": 3,
    "final_loss": float(trainer_stats.metrics['train_loss']),
    "training_time_minutes": train_time / 60,
    "date": datetime.now().isoformat(),
}

with open(f"{OUTPUT_DIR}/training_info.json", 'w') as f:
    json.dump(training_info, f, indent=2)

print("\n" + "="*70)
print("🎉 MEDICAL MODEL COMPLETE!")
print("="*70)
print(f"\n📦 Outputs:")
print(f"   • {OUTPUT_DIR}_merged_4bit/ (use this!)")
print(f"\n💡 Next: Upload to HuggingFace")
print(f"   huggingface-cli upload YOUR_USERNAME/medical-encyclopedia {OUTPUT_DIR}_merged_4bit")

# 8. Quick Test
print("\n" + "="*70)
print("🧪 TESTING MODEL")
print("="*70 + "\n")

FastLanguageModel.for_inference(model)

test_q = "What is diabetes?"
prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>

{test_q}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
answer = response.split("assistant")[-1].strip()

print(f"Q: {test_q}")
print(f"A: {answer}\n")
print("✅ Model is working perfectly!")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-12-26 15:57:31.105915: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766764651.305214      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766764651.361613      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766764651.836845      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766764651.836887      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766764651.836890      55 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
✅ All packages loaded successfully!

✅ You have 1091 Q&A pairs ready to train!

FINE-TUNING LLAMA 3.1 8B ON YOUR MEDICAL DATA

📥 Loading model...
==((====))==  Unsloth 2025.12.9: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

... (7 outputs left out)

✓ Loaded 1,091 examples
🔄 Formatting...


Map:   0%|          | 0/1091 [00:00<?, ? examples/s]

✓ Formatted

⚙️  Configuring trainer...


NameError: name 'psutil' is not defined

In [3]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig  # Changed from TrainingArguments to SFTConfig
from transformers import DataCollatorForSeq2Seq
import time
import json
from datetime import datetime

# ==========================================
# PART 2: FINE-TUNE WITH GENERATED DATA
# ==========================================

# --- 1. SETUP VARIABLES (Prevents NameErrors) ---
MODEL_PATH = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"
OUTPUT_DIR = "medical_llama3.1_8b_finetuned"  # <--- Defined here to prevent crash
# Dummy values for variables lost from Part 1 (so script doesn't crash)
chunks_to_process = "N/A (Kernel Restarted)"
generation_time = 0 
start_time = time.time()

print("="*70)
print("PART 2: FINE-TUNING LLAMA 3.1 8B ON MEDICAL ENCYCLOPEDIA")
print("="*70)

# --- 2. RELOAD MODEL ---
print("\n📥 Loading model for training...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    device_map = "auto",
)
print("✓ Model loaded")

# --- 3. CONFIGURE LORA ---
print("\n⚙️  Configuring LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,
)
print("✓ LoRA configured")

# --- 4. LOAD DATASET ---
print(f"\n📊 Loading dataset...")
# Ensure this path matches your actual file location
data = load_dataset("json", data_files="/kaggle/input/medical-finetune-data/medical_finetune_data_final.json", split="train")
print(f"✓ Loaded {len(data):,} training examples")

def format_llama_chat(examples):
    texts = []
    for instr, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a medical assistant trained on the Gale Encyclopedia of Medicine, providing accurate evidence-based information.<|eot_id|><|start_header_id|>user<|end_header_id|>

{instr}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{out}<|eot_id|>"""
        texts.append(text)
    return {"text": texts}

print("🔄 Formatting dataset...")
data = data.map(format_llama_chat, batched=True, remove_columns=data.column_names)
print("✓ Dataset formatted")

# --- 5. CONFIGURE TRAINER ---
print("\n⚙️  Configuring trainer...")

if len(data) < 500:
    num_epochs = 5
elif len(data) < 2000:
    num_epochs = 3
else:
    num_epochs = 2

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = data,
    dataset_text_field = "text",
    max_seq_length = 2048,
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc = 4, # Explicitly set to avoid psutil error
    packing = False,
    # Use SFTConfig instead of TrainingArguments
    args = SFTConfig(
        dataset_num_proc = 4, # Double safety for the psutil error
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        num_train_epochs = num_epochs,
        warmup_ratio = 0.03,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        max_grad_norm = 0.3,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        save_strategy = "epoch",
        save_total_limit = 2,
        output_dir = "outputs",
        report_to = "none",
        seed = 3407,
    ),
)

print("✓ Trainer configured")

# --- 6. TRAIN ---
print("\n" + "="*70)
print("🚀 STARTING TRAINING")
print("="*70 + "\n")

train_start = time.time()
trainer_stats = trainer.train()
train_time = time.time() - train_start

print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)

# --- 7. SAVE MODELS ---
print(f"\n💾 Saving models to {OUTPUT_DIR}...")

# Save LoRA
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save Merged 4-bit (Best for inference)
print("   Saving merged 4-bit model (this may take a minute)...")
model.save_pretrained_merged(
    f"{OUTPUT_DIR}_merged_4bit",
    tokenizer,
    save_method = "merged_4bit"
)

# Save metadata
training_info = {
    "base_model": MODEL_PATH,
    "training_examples": len(data),
    "epochs": num_epochs,
    "final_loss": float(trainer_stats.metrics['train_loss']),
    "training_time_minutes": train_time / 60,
    "date_trained": datetime.now().isoformat(),
}

with open(f"{OUTPUT_DIR}/training_info.json", 'w') as f:
    json.dump(training_info, f, indent=2)

print("\n🎉 ALL DONE! Your model is ready.")

PART 2: FINE-TUNING LLAMA 3.1 8B ON MEDICAL ENCYCLOPEDIA

📥 Loading model for training...
==((====))==  Unsloth 2025.12.9: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

... (6 outputs left out)

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,3.176800
20,1.777900
30,1.522200
40,1.516600
50,1.516700
60,1.500000
70,1.397700
80,1.111000
90,1.031900
100,1.050200



✅ TRAINING COMPLETE!

💾 Saving models to medical_llama3.1_8b_finetuned...
   Saving merged 4-bit model (this may take a minute)...


RuntimeError: Unsloth: Merging into 4bit will cause your model to lose accuracy if you plan
to merge to GGUF or others later on. I suggest you to do this as a final step
if you're planning to do multiple saves.
If you are certain, change `save_method` to `merged_4bit_forced`.

In [4]:
# ==========================================
# 7. SAVE MODELS (CORRECTED)
# ==========================================
print(f"\n💾 Saving models to {OUTPUT_DIR}...")

# 1. Save LoRA adapters (Safety backup)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✓ LoRA adapters saved.")

# 2. Save Merged 16-bit (Good for GGUF conversion later)
print("   Saving merged 16-bit model...")
model.save_pretrained_merged(
    f"{OUTPUT_DIR}_merged_16bit",
    tokenizer,
    save_method = "merged_16bit",
)
print("✓ Merged 16-bit saved.")

# 3. Save Merged 4-bit (The one causing error - FIXED)
print("   Saving merged 4-bit model (forced)...")
model.save_pretrained_merged(
    f"{OUTPUT_DIR}_merged_4bit",
    tokenizer,
    save_method = "merged_4bit_forced",  # <--- CHANGED THIS LINE
)
print("✓ Merged 4-bit saved.")

print("\n🎉 ALL DONE! Your model is safely saved.")


💾 Saving models to medical_llama3.1_8b_finetuned...
✓ LoRA adapters saved.
   Saving merged 16-bit model...


config.json:   0%|          | 0.00/947 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

... (8 outputs left out)

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [02:10<00:00, 32.54s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/medical_llama3.1_8b_finetuned_merged_16bit`
✓ Merged 16-bit saved.
   Saving merged 4-bit model (forced)...


RuntimeError: Unsloth: Failed saving locally - no disk space left. Uploading can work luckily! Use .push_to_hub instead.

In [9]:
import shutil
import os

# Define your output directory matches previous steps
OUTPUT_DIR = "medical_llama3.1_8b_finetuned"

print("🚨 OPERATING ON DISK STORAGE...")

# 1. Delete the huge 16-bit model (~16GB)
dir_16bit = f"{OUTPUT_DIR}_merged_16bit"

if os.path.exists(dir_16bit):
    print(f"🗑️ Deleting 16-bit model at: {dir_16bit}...")
    shutil.rmtree(dir_16bit)
    print("✓ Disk space freed (approx 16GB recovered).")
else:
    print("ℹ️ 16-bit folder not found (already deleted?).")

# 2. Save the efficient 4-bit model (~5.5GB)
print(f"\n💾 Saving merged 4-bit model to {OUTPUT_DIR}_merged_4bit...")
print("   (This takes about 2-3 minutes)...")

model.save_pretrained_merged(
    f"{OUTPUT_DIR}_merged_4bit",
    tokenizer,
    save_method = "merged_4bit_forced", # Forced allows direct 4-bit saving
)

print("\n✅ SUCCESS! Your RAG-ready model is saved at:")
print(f"   /kaggle/working/{OUTPUT_DIR}_merged_4bit")

🚨 OPERATING ON DISK STORAGE...
ℹ️ 16-bit folder not found (already deleted?).

💾 Saving merged 4-bit model to medical_llama3.1_8b_finetuned_merged_4bit...
   (This takes about 2-3 minutes)...
Unsloth: Merging LoRA weights into 4bit model...
Unsloth: Merging finished.
Unsloth: Found skipped modules: ['lm_head']. Updating config.
Unsloth: Saving merged 4bit model to medical_llama3.1_8b_finetuned_merged_4bit...
Unsloth: Merged 4bit model saved.
Unsloth: Merged 4bit model process completed.

✅ SUCCESS! Your RAG-ready model is saved at:
   /kaggle/working/medical_llama3.1_8b_finetuned_merged_4bit


In [10]:
import os

# 1. Zip the LoRA Adapters (Small ~200MB)
print("📦 Zipping LoRA adapters...")
os.system("zip -r lora_adapters.zip medical_llama3.1_8b_finetuned")
print("✓ Created lora_adapters.zip")

# 2. Zip the Merged 4-bit Model (Large ~5.5GB)
# This might take 2-3 minutes due to size
print("📦 Zipping Merged 4-bit model (this will take a moment)...")
os.system("zip -r merged_4bit_model.zip medical_llama3.1_8b_finetuned_merged_4bit")
print("✓ Created merged_4bit_model.zip")

print("\n🎉 Zipping complete!")

📦 Zipping LoRA adapters...
  adding: medical_llama3.1_8b_finetuned/ (stored 0%)
  adding: medical_llama3.1_8b_finetuned/README.md (deflated 65%)
  adding: medical_llama3.1_8b_finetuned/special_tokens_map.json (deflated 71%)
  adding: medical_llama3.1_8b_finetuned/tokenizer_config.json (deflated 96%)
  adding: medical_llama3.1_8b_finetuned/adapter_model.safetensors (deflated 7%)
  adding: medical_llama3.1_8b_finetuned/tokenizer.json (deflated 85%)
  adding: medical_llama3.1_8b_finetuned/adapter_config.json (deflated 56%)
✓ Created lora_adapters.zip
📦 Zipping Merged 4-bit model (this will take a moment)...
  adding: medical_llama3.1_8b_finetuned_merged_4bit/ (stored 0%)
  adding: medical_llama3.1_8b_finetuned_merged_4bit/generation_config.json (deflated 36%)
  adding: medical_llama3.1_8b_finetuned_merged_4bit/model-00002-of-00002.safetensors (deflated 23%)
  adding: medical_llama3.1_8b_finetuned_merged_4bit/model.safetensors.index.json (deflated 96%)
  adding: medical_llama3.1_8b_finetun